# 향후 희망 여가활동 선호 순위모형 Baseline
- 2024~2025년 향후 희망 여가활동 1~3순위를 이용해 문화시설 선호 순위 테이블을 생성함.
- 만족 여가활동 모델과 동일한 문화누리 중분류 매핑 기준을 적용함.
- ROL과 다항 로지스틱 baseline 성능을 비교함.

## 분석 환경 및 데이터 경로 설정
- 분석에 필요한 패키지와 경로를 설정함.
- 원자료와 기존 중분류 매핑표를 불러올 준비를 수행함.

In [ ]:
import pathlib
import numpy as np
import pandas as pd

from pathlib import Path
from scipy.optimize import minimize
from scipy.special import logsumexp
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, f1_score, balanced_accuracy_score, precision_recall_fscore_support

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

BASE_PATH = pathlib.Path().resolve()

PROJECT_PATH = None
for path in [BASE_PATH, *BASE_PATH.parents]:
    if (path / "notebooks" / "preference" / "data").exists():
        PROJECT_PATH = path
        break

if PROJECT_PATH is None:
    raise FileNotFoundError("???? ??? ?? ?????.")

PREFERENCE_PATH = PROJECT_PATH / "notebooks" / "preference"
SOURCE_PATH = PREFERENCE_PATH / "data" / "source"
PROCESSED_PATH = PREFERENCE_PATH / "data" / "processed" / "satisfaction"

SURVEY_PATH = SOURCE_PATH / "leisure_activity_survey_2021_2025_selected_columns.csv"
MAPPING_PATH = PROCESSED_PATH / "ml_activity_category_mapping.csv"

print("PROJECT_PATH:", PROJECT_PATH)
print("SURVEY_PATH 존재:", SURVEY_PATH.exists())
print("MAPPING_PATH 존재:", MAPPING_PATH.exists())

## 데이터 불러오기 및 기본 구조 확인
- 국민여가활동조사 선택 칼럼 원자료를 불러옴.
- 2024~2025년에 향후 희망 여가활동 응답이 존재하는지 확인함.
- 기존 문화누리 중분류 매핑표를 불러옴.

In [ ]:
survey = pd.read_csv(SURVEY_PATH, encoding="utf-8-sig", low_memory=False)
activity_mapping = pd.read_csv(MAPPING_PATH, encoding="utf-8-sig")

activity_mapping["활동코드"] = pd.to_numeric(
    activity_mapping["활동코드"],
    errors="coerce",
).astype("Int64")

future_rank_cols_raw = [
    "향후 희망하는 여가활동 1순위",
    "향후 희망하는 여가활동 2순위",
    "향후 희망하는 여가활동 3순위",
]

satisfaction_rank_cols_raw = [
    "가장 만족스러운 여가활동 1순위",
    "가장 만족스러운 여가활동 2순위",
    "가장 만족스러운 여가활동 3순위",
]

future_notna_by_year = survey.groupby("조사년도")[future_rank_cols_raw].agg(lambda x: x.notna().sum())

valid_categories = (
    activity_mapping
    .loc[activity_mapping["학습타깃사용여부"], "중분류"]
    .drop_duplicates()
    .sort_values()
    .tolist()
)

print("survey 구조:", survey.shape)
print("activity_mapping 구조:", activity_mapping.shape)
print("학습 대상 중분류:", valid_categories)
print("향후 희망 응답 연도별 비결측")
display(future_notna_by_year)

display(activity_mapping.head(20))

## 향후 희망 문항 Value 점검
- 만족 문항과 향후 희망 문항의 활동코드 집합을 비교함.
- 향후 희망 문항에만 존재하거나 기존 매핑표에 없는 활동코드를 확인함.

In [ ]:
def make_activity_long(data, columns, label):
    result_list = []
    
    for rank, col in enumerate(columns, start=1):
        temp = data[["조사년도", col]].copy()
        temp = temp.rename(columns={col: "활동코드"})
        temp["순위"] = rank
        temp["문항"] = label
        temp["활동코드"] = pd.to_numeric(temp["활동코드"], errors="coerce")
        temp = temp[temp["활동코드"].notna()].copy()
        temp["활동코드"] = temp["활동코드"].astype(int)
        result_list.append(temp)
    
    return pd.concat(result_list, ignore_index=True)

survey_2425 = survey[survey["조사년도"].isin([2024, 2025])].copy()

future_long_raw = make_activity_long(survey_2425, future_rank_cols_raw, "향후희망")
satisfaction_long_raw = make_activity_long(survey_2425, satisfaction_rank_cols_raw, "만족")

future_codes = set(future_long_raw["활동코드"].unique())
satisfaction_codes = set(satisfaction_long_raw["활동코드"].unique())
mapping_codes = set(activity_mapping["활동코드"].dropna().astype(int).unique())

future_only_codes = sorted(future_codes - satisfaction_codes)
satisfaction_only_codes = sorted(satisfaction_codes - future_codes)
unmapped_future_codes = sorted(future_codes - mapping_codes)

print("2024~2025 향후 희망 고유 활동코드 수:", len(future_codes))
print("2024~2025 만족 고유 활동코드 수:", len(satisfaction_codes))
print("향후 희망에만 있는 활동코드 수:", len(future_only_codes))
print("만족에만 있는 활동코드 수:", len(satisfaction_only_codes))
print("향후 희망 매핑 누락 활동코드 수:", len(unmapped_future_codes))

code_check = pd.DataFrame({
    "구분": ["향후희망만", "만족만", "향후희망_매핑누락"],
    "활동코드": [future_only_codes, satisfaction_only_codes, unmapped_future_codes],
})

display(code_check)

## 향후 희망 순위 테이블 생성
- 2024~2025년 응답만 사용함.
- 교통수단, 숙박, 여행사, 분류범위외 등 학습 제외 분류를 제거함.
- 1~3순위 내 중복 중분류는 앞순위만 남기고 제거함.
- 제거 후 남은 유효 중분류를 1~3순위로 다시 압축함.

In [ ]:
feature_cols = [
    "조사년도",
    "성별",
    "연령",
    "17개 시도",
    "지역규모",
    "최종가중치",
    "학력",
    "가구소득",
    "장애여부",
    "여가활동을 위한 월평균 지출액",
    "적절하다고 생각하는 월평균 여가비용",
    "평일 하루 평균 여가시간",
    "휴일 하루 평균 여가시간",
    "생활권 내 공공문화여가시설 이용 충분도",
    "생활권 내 공공문화여가시설 이용 여부",
    "생활권 내 공공문화여가시설 만족도",
    "여가활동 제약요인_시간부족",
    "여가활동 제약요인_경제적 지출 부담",
    "여가활동 제약요인_여가활동 경험 부족",
    "여가활동 제약요인_여가 정보 부족",
    "여가활동 제약요인_질병 및 장애",
    "여가활동 제약요인_여가 동반자 없음",
    "여가활동 제약요인_여가시설 접근성 부족",
    "여가활동 제약요인_여가프로그램 부족",
]

mapping_info = (
    activity_mapping
    .set_index("활동코드")[["여가활동명", "중분류", "학습타깃사용여부"]]
    .to_dict("index")
)

future_base = survey_2425[feature_cols + future_rank_cols_raw].copy().reset_index(drop=True)
future_base["응답자_ID"] = [
    f"PREF_{i:05d}"
    for i in range(1, len(future_base) + 1)
]

rank_rows = []

for idx, row in future_base.iterrows():
    valid_rank = []
    exclude_count = 0
    duplicate_count = 0
    raw_codes = []
    raw_categories = []
    
    for col in future_rank_cols_raw:
        code = pd.to_numeric(row[col], errors="coerce")
        
        if pd.isna(code):
            raw_codes.append(np.nan)
            raw_categories.append(np.nan)
            continue
        
        code = int(code)
        raw_codes.append(code)
        info = mapping_info.get(code)
        
        if info is None:
            raw_categories.append(np.nan)
            exclude_count += 1
            continue
        
        category = info["중분류"]
        raw_categories.append(category)
        
        if not bool(info["학습타깃사용여부"]):
            exclude_count += 1
            continue
        
        if category in valid_rank:
            duplicate_count += 1
            continue
        
        valid_rank.append(category)
    
    temp = {
        "응답자_ID": row["응답자_ID"],
        "향후희망_원코드_1순위": raw_codes[0],
        "향후희망_원코드_2순위": raw_codes[1],
        "향후희망_원코드_3순위": raw_codes[2],
        "향후희망_원중분류_1순위": raw_categories[0],
        "향후희망_원중분류_2순위": raw_categories[1],
        "향후희망_원중분류_3순위": raw_categories[2],
        "향후희망_유효순위수": len(valid_rank),
        "제외분류_제거수": exclude_count,
        "중복중분류_제거수": duplicate_count,
    }
    
    for rank in range(3):
        temp[f"향후희망_유효중분류_{rank + 1}순위"] = valid_rank[rank] if rank < len(valid_rank) else np.nan
    
    rank_rows.append(temp)

future_rank_info = pd.DataFrame(rank_rows)
future_rank_base = future_base.drop(columns=future_rank_cols_raw).merge(
    future_rank_info,
    on="응답자_ID",
    how="left",
)

rank_cols = [
    "향후희망_유효중분류_1순위",
    "향후희망_유효중분류_2순위",
    "향후희망_유효중분류_3순위",
]

print("future_base 구조:", future_base.shape)
print("future_rank_base 구조:", future_rank_base.shape)
print("유효순위수 분포")
print(future_rank_base["향후희망_유효순위수"].value_counts(dropna=False).sort_index())
print("제외분류 제거수 합:", future_rank_base["제외분류_제거수"].sum())
print("중복중분류 제거수 합:", future_rank_base["중복중분류_제거수"].sum())
print("학습 가능한 응답자 수:", future_rank_base[rank_cols[0]].notna().sum())

display(future_rank_base.head())

## 중분류 분포 확인
- 향후 희망 유효 중분류의 순위별 분포를 확인함.
- 1순위와 Top-3 포함률을 확인함.

In [ ]:
rank_distribution_list = []

for rank, col in enumerate(rank_cols, start=1):
    temp = future_rank_base[col].value_counts(dropna=False).reset_index()
    temp.columns = ["중분류", "빈도"]
    temp["순위"] = rank
    temp["비율"] = temp["빈도"] / len(future_rank_base)
    rank_distribution_list.append(temp)

rank_distribution = pd.concat(rank_distribution_list, ignore_index=True)
rank_distribution = rank_distribution[["순위", "중분류", "빈도", "비율"]]

display(rank_distribution)

top3_rows = []

for category in valid_categories:
    include_count = future_rank_base[rank_cols].eq(category).any(axis=1).sum()
    top3_rows.append({
        "중분류": category,
        "Top3_포함응답자수": include_count,
        "Top3_포함률": include_count / len(future_rank_base),
    })

top3_include = pd.DataFrame(top3_rows).sort_values("Top3_포함률", ascending=False)

display(top3_include)

## 코드 라벨 및 입력 변수 확정
- 성별, 연령대, 시도, 지역규모 라벨을 생성함.
- 기존 baseline과 동일한 직접 대응 입력변수를 사용함.
- 입력 변수 결측을 확인함.

In [ ]:
sex_map = {
    1: "남성",
    2: "여성",
}

age_map = {
    1: "15-19세",
    2: "20대",
    3: "30대",
    4: "40대",
    5: "50대",
    6: "60대",
    7: "70세 이상",
}

sido_map = {
    1: "서울",
    2: "부산",
    3: "대구",
    4: "인천",
    5: "광주",
    6: "대전",
    7: "울산",
    8: "세종",
    9: "경기",
    10: "강원",
    11: "충북",
    12: "충남",
    13: "전북",
    14: "전남",
    15: "경북",
    16: "경남",
    17: "제주",
}

region_size_map = {
    1: "대도시",
    2: "중소도시",
    3: "읍면지역",
}

model_df = future_rank_base[future_rank_base[rank_cols[0]].notna()].copy()

model_df["성별_라벨"] = model_df["성별"].map(sex_map)
model_df["연령대"] = model_df["연령"].map(age_map)
model_df["시도"] = model_df["17개 시도"].map(sido_map)
model_df["지역규모_라벨"] = model_df["지역규모"].map(region_size_map)
model_df["조사년도_라벨"] = model_df["조사년도"].astype(str)

model_feature_cols = [
    "성별_라벨",
    "연령대",
    "시도",
    "지역규모_라벨",
    "조사년도_라벨",
]

model_missing = (
    model_df[model_feature_cols + rank_cols + ["최종가중치"]]
    .isna()
    .sum()
    .reset_index(name="결측치")
    .rename(columns={"index": "칼럼명"})
)
model_missing["결측률"] = model_missing["결측치"] / len(model_df)

print("모델링 대상 구조:", model_df.shape)
print("입력 변수:", model_feature_cols)
display(model_missing)

print("1순위 분포")
display(model_df[rank_cols[0]].value_counts(normalize=True).reset_index().rename(columns={"index": "중분류", rank_cols[0]: "비율"}))

## Train / Valid / Test 분리
- 70:15:15 비율로 데이터를 분리함.
- 조사년도와 1순위 중분류 분포가 유지되도록 stratify를 적용함.

In [ ]:
split_key = (
    model_df["조사년도_라벨"]
    + "_"
    + model_df[rank_cols[0]]
)

train_valid_idx, test_idx = train_test_split(
    model_df.index,
    test_size=0.15,
    random_state=42,
    stratify=split_key,
)

train_valid_df = model_df.loc[train_valid_idx].copy()
train_valid_key = (
    train_valid_df["조사년도_라벨"]
    + "_"
    + train_valid_df[rank_cols[0]]
)

train_idx, valid_idx = train_test_split(
    train_valid_df.index,
    test_size=0.15 / 0.85,
    random_state=42,
    stratify=train_valid_key,
)

train_df = model_df.loc[train_idx].copy()
valid_df = model_df.loc[valid_idx].copy()
test_df = model_df.loc[test_idx].copy()

split_summary = pd.DataFrame({
    "데이터": ["train", "valid", "test"],
    "응답자수": [len(train_df), len(valid_df), len(test_df)],
    "비율": [len(train_df) / len(model_df), len(valid_df) / len(model_df), len(test_df) / len(model_df)],
})

display(split_summary)

split_label_df = pd.concat([
    train_df.assign(split="train"),
    valid_df.assign(split="valid"),
    test_df.assign(split="test"),
])

display(pd.crosstab(
    split_label_df[rank_cols[0]],
    split_label_df["split"],
    normalize="columns",
))

## Rank-Ordered Logit 학습 배열 생성
- 범주형 입력변수를 더미 변수로 변환함.
- 1~3순위 중분류를 정수 인덱스 배열로 변환함.
- 최종가중치를 평균 1로 정규화함.

In [ ]:
all_feature_df = pd.get_dummies(
    model_df[model_feature_cols],
    drop_first=True,
    dtype=float,
)

all_feature_df.insert(0, "상수", 1.0)
feature_columns = all_feature_df.columns.tolist()

category_to_idx = {
    category: idx
    for idx, category in enumerate(valid_categories)
}

idx_to_category = {
    idx: category
    for category, idx in category_to_idx.items()
}

reference_category = "영상"
reference_idx = category_to_idx[reference_category]
nonref_idx = [
    idx for idx in range(len(valid_categories))
    if idx != reference_idx
]


def make_model_arrays(data):
    X = all_feature_df.loc[data.index].to_numpy(dtype=float)
    y = np.full((len(data), 3), -1, dtype=int)
    
    for rank, col in enumerate(rank_cols):
        y[:, rank] = (
            data[col]
            .map(category_to_idx)
            .fillna(-1)
            .astype(int)
            .to_numpy()
        )
    
    sample_weight = data["최종가중치"].to_numpy(dtype=float)
    sample_weight = sample_weight / np.nanmean(sample_weight)
    
    return X, y, sample_weight

X_train, y_train, weight_train = make_model_arrays(train_df)
X_valid, y_valid, weight_valid = make_model_arrays(valid_df)
X_test, y_test, weight_test = make_model_arrays(test_df)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("X_test:", X_test.shape)
print("입력 변수 더미 수:", len(feature_columns))
print("기준 중분류:", reference_category)

## Rank-Ordered Logit 모델 학습
- 1~3순위 순서 정보를 이용해 ROL 모델을 학습함.
- L-BFGS-B 최적화와 약한 L2 정규화를 적용함.

In [ ]:
K = len(valid_categories)
D = X_train.shape[1]
l2_alpha = 1e-4

choice_weight_sum = sum(
    weight_train[y_train[:, rank] >= 0].sum()
    for rank in range(3)
)


def unpack_params(params):
    W_nonref = params.reshape(D, K - 1)
    W = np.zeros((D, K), dtype=float)
    W[:, nonref_idx] = W_nonref
    return W


def loss_grad(params):
    W = unpack_params(params)
    scores = X_train @ W
    grad_scores = np.zeros_like(scores)
    loss = 0.0
    
    for rank in range(3):
        valid_mask = y_train[:, rank] >= 0
        
        if valid_mask.sum() == 0:
            continue
        
        valid_idx = np.where(valid_mask)[0]
        chosen = y_train[valid_idx, rank]
        
        masked_scores = scores[valid_idx].copy()
        remain_mask = np.ones_like(masked_scores, dtype=bool)
        
        for prev_rank in range(rank):
            prev_chosen = y_train[valid_idx, prev_rank]
            remain_mask[np.arange(len(valid_idx)), prev_chosen] = False
        
        masked_scores[~remain_mask] = -np.inf
        log_den = logsumexp(masked_scores, axis=1)
        probs = np.exp(masked_scores - log_den[:, None])
        probs[~remain_mask] = 0
        
        row_grad = probs
        row_grad[np.arange(len(valid_idx)), chosen] -= 1
        
        row_weight = weight_train[valid_idx]
        loss += np.sum(row_weight * (log_den - scores[valid_idx, chosen]))
        grad_scores[valid_idx] += row_grad * row_weight[:, None]
    
    grad_full = X_train.T @ grad_scores
    grad_nonref = grad_full[:, nonref_idx]
    params_nonref = W[:, nonref_idx].ravel()
    
    loss = loss / choice_weight_sum + 0.5 * l2_alpha * np.sum(params_nonref ** 2)
    grad = (grad_nonref / choice_weight_sum).ravel() + l2_alpha * params_nonref
    
    return loss, grad

init_params = np.zeros(D * (K - 1), dtype=float)

rol_result = minimize(
    fun=lambda params: loss_grad(params),
    x0=init_params,
    jac=True,
    method="L-BFGS-B",
    options={"maxiter": 500},
)

W_hat = unpack_params(rol_result.x)

print("수렴 여부:", rol_result.success)
print("수렴 메시지:", rol_result.message)
print("반복 횟수:", rol_result.nit)
print("최종 목적함수:", rol_result.fun)

## 성능 평가 함수 정의
- Top1, Top3, MRR, NDCG@3, LogLoss를 산출함.
- 다항 로지스틱 비교를 위해 확률 행렬 기반 평가 함수도 생성함.

In [ ]:
def predict_prob(X, W):
    scores = X @ W
    scores = scores - scores.max(axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    return exp_scores / exp_scores.sum(axis=1, keepdims=True)


def rol_log_loss(X, y, W, sample_weight=None):
    if sample_weight is None:
        sample_weight = np.ones(X.shape[0])
    
    scores = X @ W
    total_loss = 0.0
    total_weight = 0.0
    
    for rank in range(3):
        valid_mask = y[:, rank] >= 0
        
        if valid_mask.sum() == 0:
            continue
        
        valid_idx = np.where(valid_mask)[0]
        chosen = y[valid_idx, rank]
        
        masked_scores = scores[valid_idx].copy()
        remain_mask = np.ones_like(masked_scores, dtype=bool)
        
        for prev_rank in range(rank):
            prev_chosen = y[valid_idx, prev_rank]
            remain_mask[np.arange(len(valid_idx)), prev_chosen] = False
        
        masked_scores[~remain_mask] = -np.inf
        log_den = logsumexp(masked_scores, axis=1)
        
        row_weight = sample_weight[valid_idx]
        total_loss += np.sum(row_weight * (log_den - scores[valid_idx, chosen]))
        total_weight += row_weight.sum()
    
    return total_loss / total_weight


def ndcg_at_3(prob, y):
    pred_order = np.argsort(-prob, axis=1)[:, :3]
    ndcg_list = []
    
    for i in range(len(y)):
        relevance = {}
        
        for rank in range(3):
            if y[i, rank] >= 0:
                relevance[y[i, rank]] = 3 - rank
        
        if len(relevance) == 0:
            continue
        
        dcg = 0.0
        
        for position, category_idx in enumerate(pred_order[i], start=1):
            rel = relevance.get(category_idx, 0)
            dcg += (2 ** rel - 1) / np.log2(position + 1)
        
        ideal_relevance = sorted(relevance.values(), reverse=True)[:3]
        idcg = sum(
            (2 ** rel - 1) / np.log2(position + 1)
            for position, rel in enumerate(ideal_relevance, start=1)
        )
        
        ndcg_list.append(dcg / idcg if idcg > 0 else np.nan)
    
    return np.nanmean(ndcg_list)


def evaluate_rank_model(data_name, X, y, W):
    prob = predict_prob(X, W)
    pred_order = np.argsort(-prob, axis=1)
    rank1 = y[:, 0]
    
    top1_accuracy = np.mean(pred_order[:, 0] == rank1)
    top3_hit_rate = np.mean([
        rank1[i] in pred_order[i, :3]
        for i in range(len(rank1))
    ])
    mrr = np.mean([
        1 / (np.where(pred_order[i] == rank1[i])[0][0] + 1)
        for i in range(len(rank1))
    ])
    ndcg = ndcg_at_3(prob, y)
    logloss = rol_log_loss(X, y, W)
    
    return {
        "데이터": data_name,
        "Top1_Accuracy": top1_accuracy,
        "Top3_HitRate": top3_hit_rate,
        "MRR": mrr,
        "NDCG@3": ndcg,
        "ROL_LogLoss": logloss,
    }


def align_mnl_prob(model, X_mnl):
    raw_prob = model.predict_proba(X_mnl)
    prob = np.zeros((X_mnl.shape[0], len(valid_categories)))
    class_to_col = {
        category: idx
        for idx, category in enumerate(model.classes_)
    }
    
    for j, category in enumerate(valid_categories):
        if category in class_to_col:
            prob[:, j] = raw_prob[:, class_to_col[category]]
    
    row_sum = prob.sum(axis=1, keepdims=True)
    prob = np.divide(
        prob,
        row_sum,
        out=np.zeros_like(prob),
        where=row_sum > 0,
    )
    
    return prob


def evaluate_prob_rank_model(data_name, prob, y, y_label):
    pred_order = np.argsort(-prob, axis=1)
    rank1 = y[:, 0]
    
    top1_accuracy = np.mean(pred_order[:, 0] == rank1)
    top3_hit_rate = np.mean([
        rank1[i] in pred_order[i, :3]
        for i in range(len(rank1))
    ])
    mrr = np.mean([
        1 / (np.where(pred_order[i] == rank1[i])[0][0] + 1)
        for i in range(len(rank1))
    ])
    ndcg = ndcg_at_3(prob, y)
    rank1_logloss = log_loss(
        y_label,
        prob,
        labels=valid_categories,
    )
    
    return {
        "데이터": data_name,
        "Top1_Accuracy": top1_accuracy,
        "Top3_HitRate": top3_hit_rate,
        "MRR": mrr,
        "NDCG@3": ndcg,
        "Rank1_LogLoss": rank1_logloss,
    }

## ROL 성능 평가
- prior 기준모형과 ROL 학습모형을 비교함.
- prior는 train 데이터의 1순위 중분류 분포만 사용함.

In [ ]:
train_prior = train_df[rank_cols[0]].value_counts(normalize=True)

prior_prob = np.array([
    train_prior.get(category, 0)
    for category in valid_categories
])

prior_prob = prior_prob / prior_prob.sum()
prior_scores = np.log(prior_prob + 1e-12)

W_prior = np.zeros_like(W_hat)
W_prior[0, :] = prior_scores - prior_scores[reference_idx]

rol_performance_table = pd.DataFrame([
    evaluate_rank_model("train_model", X_train, y_train, W_hat),
    evaluate_rank_model("valid_model", X_valid, y_valid, W_hat),
    evaluate_rank_model("test_model", X_test, y_test, W_hat),
    evaluate_rank_model("train_prior", X_train, y_train, W_prior),
    evaluate_rank_model("valid_prior", X_valid, y_valid, W_prior),
    evaluate_rank_model("test_prior", X_test, y_test, W_prior),
]).round(4)

display(rol_performance_table)

## 다항 로지스틱 학습 및 성능 평가
- 1순위 중분류를 목표변수로 다항 로지스틱을 학습함.
- ROL과 동일한 입력변수, 데이터 분할, 최종가중치를 사용함.

In [ ]:
mnl_feature_df = all_feature_df.drop(columns="상수").copy()
mnl_feature_columns = mnl_feature_df.columns.tolist()

X_train_mnl = mnl_feature_df.loc[train_df.index].to_numpy(dtype=float)
X_valid_mnl = mnl_feature_df.loc[valid_df.index].to_numpy(dtype=float)
X_test_mnl = mnl_feature_df.loc[test_df.index].to_numpy(dtype=float)

y_train_mnl = train_df[rank_cols[0]].to_numpy()
y_valid_mnl = valid_df[rank_cols[0]].to_numpy()
y_test_mnl = test_df[rank_cols[0]].to_numpy()

mnl_model = LogisticRegression(
    solver="lbfgs",
    max_iter=1000,
    C=1.0,
)

mnl_model.fit(
    X_train_mnl,
    y_train_mnl,
    sample_weight=weight_train,
)

mnl_prob_train = align_mnl_prob(mnl_model, X_train_mnl)
mnl_prob_valid = align_mnl_prob(mnl_model, X_valid_mnl)
mnl_prob_test = align_mnl_prob(mnl_model, X_test_mnl)

mnl_prior_train = np.tile(prior_prob, (len(train_df), 1))
mnl_prior_valid = np.tile(prior_prob, (len(valid_df), 1))
mnl_prior_test = np.tile(prior_prob, (len(test_df), 1))

mnl_performance_table = pd.DataFrame([
    evaluate_prob_rank_model("train_model", mnl_prob_train, y_train, y_train_mnl),
    evaluate_prob_rank_model("valid_model", mnl_prob_valid, y_valid, y_valid_mnl),
    evaluate_prob_rank_model("test_model", mnl_prob_test, y_test, y_test_mnl),
    evaluate_prob_rank_model("train_prior", mnl_prior_train, y_train, y_train_mnl),
    evaluate_prob_rank_model("valid_prior", mnl_prior_valid, y_valid, y_valid_mnl),
    evaluate_prob_rank_model("test_prior", mnl_prior_test, y_test, y_test_mnl),
]).round(4)

print("수렴 반복 횟수:", mnl_model.n_iter_[0])
display(mnl_performance_table)

## 모델 성능 비교
- ROL과 다항 로지스틱 성능을 하나의 표로 비교함.
- LogLoss는 ROL은 순위 기준, 다항 로지스틱은 1순위 기준으로 산출함.

In [ ]:
rol_compare = rol_performance_table.copy().rename(columns={"ROL_LogLoss": "LogLoss"})
rol_compare["모델"] = "ROL"
rol_compare["LogLoss_기준"] = "순위"

mnl_compare = mnl_performance_table.copy().rename(columns={"Rank1_LogLoss": "LogLoss"})
mnl_compare["모델"] = "다항로지스틱"
mnl_compare["LogLoss_기준"] = "1순위"

model_performance_compare = pd.concat([
    rol_compare,
    mnl_compare,
], ignore_index=True)

model_performance_compare = model_performance_compare[[
    "모델",
    "데이터",
    "Top1_Accuracy",
    "Top3_HitRate",
    "MRR",
    "NDCG@3",
    "LogLoss",
    "LogLoss_기준",
]]

model_performance_compare = model_performance_compare.round(4)
display(model_performance_compare)

print("검증셋 성능")
display(model_performance_compare[model_performance_compare["데이터"].str.contains("valid")])

print("테스트셋 성능")
display(model_performance_compare[model_performance_compare["데이터"].str.contains("test")])

## 예측 분포 점검
- 검증셋의 실제 1순위 분포와 모델별 예측 1순위 분포를 비교함.
- 특정 중분류로 예측이 쏠리는지 확인함.

In [ ]:
def make_prediction_distribution(prob, data_name):
    pred_idx = np.argmax(prob, axis=1)
    pred_label = [idx_to_category[idx] for idx in pred_idx]
    
    return (
        pd.Series(pred_label)
        .value_counts(normalize=True)
        .rename_axis("중분류")
        .reset_index(name=data_name)
    )

actual_valid_distribution = (
    valid_df[rank_cols[0]]
    .value_counts(normalize=True)
    .rename_axis("중분류")
    .reset_index(name="실제_1순위_비율")
)

rol_valid_distribution = make_prediction_distribution(
    predict_prob(X_valid, W_hat),
    "ROL_예측_1순위_비율",
)

mnl_valid_distribution = make_prediction_distribution(
    mnl_prob_valid,
    "다항로지스틱_예측_1순위_비율",
)

prediction_distribution_compare = (
    actual_valid_distribution
    .merge(rol_valid_distribution, on="중분류", how="outer")
    .merge(mnl_valid_distribution, on="중분류", how="outer")
    .fillna(0)
)

prediction_distribution_compare = prediction_distribution_compare.sort_values(
    "실제_1순위_비율",
    ascending=False,
).round(4)

display(prediction_distribution_compare)

## 추가 분류 성능 평가
- LogLoss, Macro F1, Balanced Accuracy를 함께 산출함.
- ROL과 다항 로지스틱의 1순위 예측 기준 분류 성능을 비교함.
- 중분류별 Recall과 F1을 확인함.

In [ ]:
def make_classification_metric(model_name, split_name, prob, y_true):
    pred_idx = np.argmax(prob, axis=1)
    pred_label = np.array([idx_to_category[idx] for idx in pred_idx])
    
    return {
        "모델": model_name,
        "데이터": split_name,
        "Macro_F1": f1_score(
            y_true,
            pred_label,
            labels=valid_categories,
            average="macro",
            zero_division=0,
        ),
        "Weighted_F1": f1_score(
            y_true,
            pred_label,
            labels=valid_categories,
            average="weighted",
            zero_division=0,
        ),
        "Balanced_Accuracy": balanced_accuracy_score(y_true, pred_label),
        "Rank1_LogLoss": log_loss(
            y_true,
            prob,
            labels=valid_categories,
        ),
    }


def make_category_metric(model_name, split_name, prob, y_true):
    pred_idx = np.argmax(prob, axis=1)
    pred_label = np.array([idx_to_category[idx] for idx in pred_idx])
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        pred_label,
        labels=valid_categories,
        zero_division=0,
    )
    pred_count = (
        pd.Series(pred_label)
        .value_counts()
        .reindex(valid_categories, fill_value=0)
        .to_numpy()
    )
    
    return pd.DataFrame({
        "모델": model_name,
        "데이터": split_name,
        "중분류": valid_categories,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "실제건수": support,
        "예측건수": pred_count,
    })

rol_prob_valid = predict_prob(X_valid, W_hat)
rol_prob_test = predict_prob(X_test, W_hat)
rol_prior_valid = predict_prob(X_valid, W_prior)
rol_prior_test = predict_prob(X_test, W_prior)

mnl_prob_valid = align_mnl_prob(mnl_model, X_valid_mnl)
mnl_prob_test = align_mnl_prob(mnl_model, X_test_mnl)
mnl_prior_valid = np.tile(prior_prob, (len(valid_df), 1))
mnl_prior_test = np.tile(prior_prob, (len(test_df), 1))

y_valid_label = valid_df[rank_cols[0]].to_numpy()
y_test_label = test_df[rank_cols[0]].to_numpy()

classification_metric_table = pd.DataFrame([
    make_classification_metric("ROL", "valid", rol_prob_valid, y_valid_label),
    make_classification_metric("ROL", "test", rol_prob_test, y_test_label),
    make_classification_metric("ROL_prior", "valid", rol_prior_valid, y_valid_label),
    make_classification_metric("ROL_prior", "test", rol_prior_test, y_test_label),
    make_classification_metric("다항로지스틱", "valid", mnl_prob_valid, y_valid_label),
    make_classification_metric("다항로지스틱", "test", mnl_prob_test, y_test_label),
    make_classification_metric("다항_prior", "valid", mnl_prior_valid, y_valid_label),
    make_classification_metric("다항_prior", "test", mnl_prior_test, y_test_label),
]).round(4)

display(classification_metric_table)

category_metric_table = pd.concat([
    make_category_metric("ROL", "valid", rol_prob_valid, y_valid_label),
    make_category_metric("ROL", "test", rol_prob_test, y_test_label),
    make_category_metric("다항로지스틱", "valid", mnl_prob_valid, y_valid_label),
    make_category_metric("다항로지스틱", "test", mnl_prob_test, y_test_label),
], ignore_index=True).round(4)

print("test 중분류별 Recall / F1")
display(
    category_metric_table[category_metric_table["데이터"] == "test"]
    .sort_values(["중분류", "모델"])
)

print("valid 중분류별 Recall / F1")
display(
    category_metric_table[category_metric_table["데이터"] == "valid"]
    .sort_values(["중분류", "모델"])
)